In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

restaurants = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/processed/restaurants_clean.csv")

taxonomy = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/reference/category_mapping.csv", encoding="cp1252")

In [2]:
taxonomy_keep = taxonomy[taxonomy["Keep"] == "Yes"]

In [3]:
mapping = taxonomy_keep.set_index("Category").to_dict("index")

In [4]:
categories = [
    c.strip()
    for c in restaurants.iloc[0]["categories"].split(",")
]

In [5]:
for c in categories:
    if c in mapping:
        print(c, "->", mapping[c])

Bubble Tea -> {'Count': 277, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Bubble Tea', 'Keep': 'Yes', 'Notes': nan}
Coffee & Tea -> {'Count': 4053, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Cafe', 'Keep': 'Yes', 'Notes': nan}
Bakeries -> {'Count': 1889, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Bakery', 'Keep': 'Yes', 'Notes': nan}


In [6]:
def extract_features(category_string, mapping):

    features = {
        "Cuisine": set(),
        "Restaurant Type": set(),
        "Experience": set()
    }

    categories = [
        category.strip()
        for category in category_string.split(",")
    ]

    for category in categories:

        if category in mapping:

            info = mapping[category]

            feature_type = info["Feature Type"]

            value = info["Standardized Value"]

            if feature_type in features and pd.notna(value):
                features[feature_type].add(value)

    return {
        key: list(value)
        for key, value in features.items()
    }

In [7]:
restaurant = restaurants.iloc[0]["categories"]

extract_features(restaurant,mapping)



{'Cuisine': [],
 'Restaurant Type': ['Bakery', 'Cafe', 'Bubble Tea'],
 'Experience': []}

In [8]:
restaurants["Extracted Features"] = restaurants["categories"].apply(
    lambda x: extract_features(x, mapping)
)

In [9]:
restaurants[["name", "Extracted Features"]].head()

,name,Extracted Features
0,St Honore Pastries,"{'Cuisine': [], 'Restaurant Type': ['Bakery', ..."
1,Sonic Drive-In,"{'Cuisine': [], 'Restaurant Type': ['Fast Food..."
2,Tsevi's Pub And Grill,"{'Cuisine': ['Italian', 'Greek', 'American'], ..."
3,Sonic Drive-In,"{'Cuisine': [], 'Restaurant Type': ['Fast Food..."
4,Vietnamese Food Truck,"{'Cuisine': ['Vietnamese'], 'Restaurant Type':..."


In [10]:
restaurants["Restaurant Type"] = restaurants["Extracted Features"].apply(
    lambda features: features["Restaurant Type"]
)

restaurants["Experience"] = restaurants["Extracted Features"].apply(
    lambda features: features["Experience"]
)

restaurants["Cuisine"] = restaurants["Extracted Features"].apply(
    lambda features: features["Cuisine"]
)

In [11]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

cuisine_mlb = mlb.fit_transform(restaurants["Cuisine"])

cuisine_mlb

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(52268, 98))

In [12]:
Cuisine = mlb.inverse_transform(cuisine_mlb)

Cuisine

[(),
 (),
 ('American', 'Greek', 'Italian'),
 (),
 ('Vietnamese',),
 ('American',),
 ('Italian',),
 ('Japanese',),
 ('Korean',),
 (),
 ('Asian Fusion',),
 ('Italian',),
 (),
 ('Japanese',),
 ('Italian',),
 (),
 (),
 (),
 ('American',),
 (),
 ('Italian',),
 ('Chinese',),
 ('American',),
 ('Italian',),
 (),
 ('American', 'Italian'),
 (),
 (),
 ('American',),
 (),
 ('American',),
 ('American',),
 ('American',),
 ('Italian',),
 ('American',),
 (),
 ('Asian Fusion', 'Japanese'),
 (),
 (),
 (),
 (),
 ('Italian',),
 (),
 ('American', 'Italian'),
 (),
 ('Cajun/Creole',),
 ('Mexican',),
 ('French', 'Mediterranean', 'Moroccan'),
 ('American',),
 (),
 (),
 ('American',),
 ('Chinese',),
 ('Italian',),
 (),
 (),
 ('Filipino',),
 ('Japanese',),
 ('Mexican',),
 ('Mexican',),
 (),
 (),
 ('Italian',),
 ('American',),
 (),
 ('Japanese',),
 (),
 (),
 (),
 (),
 ('American',),
 (),
 ('Chinese', 'Japanese', 'Thai'),
 (),
 (),
 ('Southern',),
 ('American',),
 ('Japanese',),
 ('Hawaiian',),
 ('Irish',),
 (),


In [13]:
cuisine_df = pd.DataFrame(cuisine_mlb,columns=mlb.classes_)

In [14]:
restaurants["Restaurant Type"].apply(type).value_counts()

Restaurant Type
<class 'list'>    52268
Name: count, dtype: int64

In [15]:
all_types = set()

for lst in restaurants["Restaurant Type"]:
    all_types.update(type(x) for x in lst)

all_types

{str}

In [16]:
restaurants[
    restaurants["Restaurant Type"].apply(
        lambda lst: any(not isinstance(x, str) for x in lst)
    )
][["name", "Restaurant Type"]]

,name,Restaurant Type


In [17]:
restaurant_type_mlb = mlb.fit_transform(restaurants["Restaurant Type"])

restaurant_type = mlb.inverse_transform(restaurant_type_mlb)

restaurant_type_df = pd.DataFrame(restaurant_type_mlb, columns=mlb.classes_)

restaurant_type_df

,Acai Bowls,Alcoholic Beverages,Bagels,Bakery,Barbeque,Beer,Breakfast & Brunch,Bubble Tea,Buffets,Burgers,...,Tapas/Small Plates,Tea,Teppanyaki,Tonkatsu,Vegan,Vegetarian,Waffles,Whiskey,Wine & Spirits,Wraps
0,0,0,0,1,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,1,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
52263,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52264,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52265,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
52266,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [18]:
experience_mlb = mlb.fit_transform(restaurants["Experience"])

experience = mlb.inverse_transform(experience_mlb)

experience_df = pd.DataFrame(experience_mlb, columns=mlb.classes_)

experience_df

,Beer,Lounges,Wine
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
...,...,...,...
52263,0,0,0
52264,0,0,0
52265,0,0,0
52266,0,0,0


In [19]:
numeric_features_df = restaurants[
    [
        "latitude",
        "longitude",
        "stars",
        "review_count"
    ]
]

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(numeric_features_df[["stars","review_count"]])

scaled_numeric

scaled_numeric_df = pd.DataFrame(scaled_numeric, columns = ['stars', 'review_count'])

scaled_numeric_df


,stars,review_count
0,0.584422,-0.038463
1,-1.826420,-0.430126
2,-0.620999,-0.361321
3,-2.429131,-0.408955
4,0.584422,-0.408955
...,...,...
52263,-0.620999,-0.403662
52264,0.584422,-0.287222
52265,1.187133,-0.276637
52266,1.187133,-0.387784


In [21]:
CUISINE_WEIGHT = 3.0
TYPE_WEIGHT = 2.0
EXPERIENCE_WEIGHT = 1.5
NUMERIC_WEIGHT = 1.0

In [22]:
weighted_cuisine = cuisine_df * CUISINE_WEIGHT

weighted_type = restaurant_type_df * TYPE_WEIGHT

weighted_experience = experience_df * EXPERIENCE_WEIGHT

weighted_numeric = scaled_numeric_df * NUMERIC_WEIGHT

In [23]:
recommendation_feature_matrix = pd.concat(
    [
        weighted_cuisine,
        weighted_type,
        weighted_experience,
        weighted_numeric
    ],
    axis=1
)

In [24]:
def find_restaurant(
    restaurant_name,
    city,
    restaurants
):

    selected_index = restaurants[
        (restaurants["name"] == restaurant_name) &
        (restaurants["city"] == city)
    ].index[0]

    return selected_index

In [25]:
def get_feature_vector(
    selected_index,
    recommendation_feature_matrix
):

    selected_vector = recommendation_feature_matrix.loc[
        [selected_index]
    ]

    return selected_vector

In [26]:

def calculate_similarity(
    selected_vector,
    recommendation_feature_matrix
):

    similarity_scores = cosine_similarity(
        selected_vector,
        recommendation_feature_matrix
    )

    similarity_df = pd.DataFrame(
        similarity_scores.T,
        columns=["similarity"]
    )

    return similarity_df

In [27]:
def get_top_recommendations(
    similarity_df,
    selected_index,
    top_n=10
):

    filtered_df = similarity_df.drop(selected_index)

    top_similarity_df = filtered_df.nlargest(
        top_n,
        "similarity"
    )

    return top_similarity_df

In [28]:
def format_recommendations(
    top_similarity_df,
    restaurants
):

    recommendation_table = restaurants.loc[
        top_similarity_df.index
    ]

    recommendation_table = recommendation_table[
        [
            "name",
            "city",
            "state",
            "stars",
            "review_count"
        ]
    ]

    recommendation_table = pd.concat(
        [recommendation_table, top_similarity_df],
        axis=1
    )

    return recommendation_table

In [29]:
def recommend_restaurants(
    restaurant_name,
    city,
    restaurants,
    recommendation_feature_matrix,
    top_n=10
):

    selected_index = find_restaurant(
        restaurant_name,
        city,
        restaurants
    )

    selected_vector = get_feature_vector(
        selected_index,
        recommendation_feature_matrix
    )

    similarity_df = calculate_similarity(
        selected_vector,
        recommendation_feature_matrix
    )

    top_similarity_df = get_top_recommendations(
        similarity_df,
        selected_index,
        top_n
    )

    recommendation_table = format_recommendations(
        top_similarity_df,
        restaurants
    )

    return recommendation_table

In [30]:
recommend_restaurants(
    restaurant_name="Sonic Drive-In",
    city="Nashville",
    restaurants=restaurants,
    recommendation_feature_matrix=recommendation_feature_matrix,
    top_n=10
)

,name,city,state,stars,review_count,similarity
14919,Sonic Drive-In,Hermitage,TN,1.5,10,1.000000
16286,Dairy Queen,Huntingdon Valley,PA,1.5,10,1.000000
18008,Dairy Queen Grill & Chill,Caseyville,IL,1.5,10,1.000000
8702,Dairy Queen,Glen Carbon,IL,1.5,9,0.999999
42716,Sonic Drive-In,Belleville,IL,1.5,9,0.999999
48997,Sonic Drive-In,Nashville,TN,1.5,9,0.999999
24257,Sonic Drive-In,Nashville,TN,1.5,11,0.999999
51876,Sonic Drive-In,Hendersonville,TN,1.5,11,0.999999
9991,Sonic Drive-In,Jennings,MO,1.5,8,0.999997
40671,Sonic Drive-In,Greenbrier,TN,1.5,8,0.999997
